# 04 — Models, ablation and statistics

Results from `make experiments` (or `scripts/run_experiments.py`).
The run manifest records the data hash, config, model list and git state.

**Simulated data — never report these as research findings.**

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
run_dir = ROOT / "artifacts" / "results"

summary = pd.read_csv(run_dir / "summary.csv")
statistics = pd.read_csv(run_dir / "statistics.csv")
by_mode = pd.read_csv(run_dir / "by_mode.csv")
by_task = pd.read_csv(run_dir / "by_task.csv")
manifest = json.loads((run_dir / "run_manifest.json").read_text())
manifest["runtime_seconds"], manifest["n_states"], manifest["models"]

(381.7, 4, ['svm', 'logreg', 'random_forest', 'xgboost', 'majority'])

## Leave-one-participant-out (RQ1)

No participant appears in both training and test. Mean Macro-F1 over 22 folds.

In [2]:
lopo = summary[summary["split"] == "lopo"]
display(lopo.pivot_table(index="representation", columns="model", values="macro_f1").round(3))

model,logreg,majority,random_forest,svm,xgboost
representation,,,,,
baseline,0.512,0.304,0.513,0.503,0.506
no_context,NaN,NaN,NaN,0.521,NaN
no_states,NaN,NaN,NaN,0.489,NaN
no_transitions,NaN,NaN,NaN,0.492,NaN
proposed,0.524,0.304,0.496,0.519,0.502


## Leave-one-task-out (RQ2)

Train on three computer tasks, test on the fourth. Participants overlap across
train and test, so this measures **task transfer**, not participant
generalisation. Reading is excluded (near-zero typing by design).

In [3]:
loto = summary[summary["split"] == "loto"]
display(loto.pivot_table(index="representation", columns="model", values="macro_f1").round(3))

model,logreg,majority,random_forest,svm,xgboost
representation,,,,,
baseline,0.603,0.329,0.576,0.597,0.577
no_context,NaN,NaN,NaN,0.609,NaN
no_states,NaN,NaN,NaN,0.571,NaN
no_transitions,NaN,NaN,NaN,0.513,NaN
proposed,0.538,0.329,0.604,0.550,0.569


## Ablation ladder

Same folds for every variant, SVM as the reference model. `proposed` removes
one group at a time from the full representation.

In [4]:
ablation = lopo[lopo["model"] == "svm"].set_index("representation")["macro_f1"]
display(ablation.round(3).to_frame("macro_f1"))

,macro_f1
representation,
baseline,0.503
no_context,0.521
no_states,0.489
no_transitions,0.492
proposed,0.519


## Paired statistics

Proposed vs baseline per participant (paired Wilcoxon), with bootstrap CI and
Cohen's dz. n = 22 participants.

In [5]:
display(statistics.round(3))

,baseline,proposed,n_participants,mean_baseline,mean_proposed,mean_diff,ci_low,ci_high,cohens_dz,wilcoxon_statistic,wilcoxon_p,model,contrast
0,baseline,proposed,22,0.512,0.524,0.012,-0.017,0.041,0.167,100.0,0.406,logreg,proposed_vs_baseline
1,baseline,proposed,22,0.304,0.304,0.000,0.000,0.000,NaN,NaN,NaN,majority,proposed_vs_baseline
2,baseline,proposed,22,0.513,0.496,-0.017,-0.043,0.007,-0.280,88.0,0.222,random_forest,proposed_vs_baseline
3,baseline,proposed,22,0.503,0.519,0.016,-0.010,0.043,0.238,102.0,0.443,svm,proposed_vs_baseline
4,no_context,proposed,22,0.521,0.519,-0.003,-0.023,0.020,-0.056,88.0,0.526,svm,proposed_vs_no_context
5,no_states,proposed,22,0.489,0.519,0.029,0.006,0.053,0.489,68.0,0.059,svm,proposed_vs_no_states
6,no_transitions,proposed,22,0.492,0.519,0.026,0.001,0.051,0.425,75.0,0.098,svm,proposed_vs_no_transitions
7,baseline,proposed,22,0.506,0.502,-0.005,-0.024,0.015,-0.101,112.0,0.656,xgboost,proposed_vs_baseline


## Screen vs non-screen

Pooled test predictions, not an average of fold scores. Non-screen sessions are
where aggregated activity features are weakest.

In [6]:
mode_view = by_mode[(by_mode["split"] == "lopo") & (by_mode["model"] == "svm")]
display(mode_view.pivot_table(index="study_mode", columns="representation", values="macro_f1").round(3))

representation,baseline,no_context,no_states,no_transitions,proposed
study_mode,,,,,
non-screen,0.380,0.507,0.376,0.376,0.499
screen,0.618,0.637,0.646,0.652,0.645


## How to read this

- Macro-F1 differences of a few hundredths are noise at 22 participants.
- The ablation answers *which group* contributes, not whether the whole model
  is significant.
- LOTO numbers are higher because the same participants appear on both sides;
  LOPO is the participant-level result.
- Every fold refits the scaler, HMM and imputation medians on training data.
  The state count (4) is fixed from Phase 3, not re-selected per fold.